<a href="https://colab.research.google.com/github/Jasp3r16/thesis_generative_timber/blob/main/24_25_optimizer_workflow_with_cost_and_ILP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 24_25_optimizer_workflow_with_cost_and_ILP  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-27
### Properties script
**Goal:** to generate a cost matrix for the geometry with use of the timber datasets, then using ILP to find the best matches   
**Inputs:**
*   CSV timber dataset
*   Digital geometry

**Outputs:**
*   Best match for each element in a structure

## Mounting Google drive

In [ ]:
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Definieer je paden (Pas de naam 'Thesis_Project' aan naar jouw mapnaam)
BASE_PATH = '/content/drive/MyDrive/Thesis_TU/'
DATA_PATH = os.path.join(BASE_PATH, 'data_thesis/')
SCRIPT_PATH = os.path.join(BASE_PATH, 'notebooks_thesis/')
OUTPUT_PATH = os.path.join(BASE_PATH, 'research_exports/')

# 3. Voeg de Scripts map toe aan het systeem-pad
# Hierdoor kun je .py bestanden uit die map 'importen'
if SCRIPT_PATH not in sys.path:
    sys.path.append(SCRIPT_PATH)

print(f"✅ Drive mounted. Project directory: {BASE_PATH}")

Mounted at /content/drive
✅ Drive mounted. Project directory: /content/drive/MyDrive/Thesis_TU/


# IMPORTING

In [ ]:
import pandas as pd

# Laad de geëxporteerde dataset in je nieuwe omgeving
# Voeg sep=';' toe aan je inlaad-functie!
df_input_csv = pd.read_csv(DATA_PATH + 'complete_timber.csv', sep=';')

# Print de kolommen om zeker te weten dat ze goed gesplitst zijn
print("Gevonden kolommen:", df_input_csv.columns.tolist())

# Controleer of alle data, inclusief de RS_0000 ID's en binaire states, goed is overgekomen
print("\nDataset succesvol ingeladen. Hier zijn de eerste elementen:\n")
print(df_input_csv.head())

Gevonden kolommen: ['Member_ID', 'State', 'Length_Actual', 'Depth', 'Width', 'E_modulus_eff', 'f_mk', 'Density', 'Embodied Carbon Coëfficiënt', 'Transport_Dist', 'Emmisiefactor', 'Bewerkingsfactor']

Dataset succesvol ingeladen. Hier zijn de eerste elementen:

  Member_ID  State  Length_Actual  Depth  Width  E_modulus_eff  f_mk  Density  \
0  NS_00000      0         2400.0  100.0   38.0        11000.0    24      420   
1  NS_00001      0         2400.0  100.0   50.0        11000.0    24      420   
2  NS_00002      0         2400.0  100.0   75.0        11000.0    24      420   
3  NS_00003      0         2400.0  100.0  100.0        11000.0    24      420   
4  NS_00004      0         2400.0  150.0   38.0        11000.0    24      420   

   Embodied Carbon Coëfficiënt  Transport_Dist  Emmisiefactor  \
0                        150.0             500         0.1473   
1                        150.0             500         0.1019   
2                        150.0             500         0.

# COST MATRIX

## LCA Calculation

In [ ]:
# ==========================================
# CEL: PSEUDO-LCA BEREKENING (IN-MEMORY)
# ==========================================

def calculate_pseudo_lca(df):
    """
    Berekent de pseudo-LCA score (E_cost) voor een reeds ingeladen DataFrame
    met de timber stock.
    """
    # Maak een kopie om waarschuwingen (SettingWithCopyWarning) te voorkomen
    # en de originele input dataset zuiver te houden.
    df_lca = df.copy()

    # --- LCA PARAMETERS ---
    # CO2 penalty voor het ontspijkeren/bewerken (in kg CO2eq)
    PROCESSING_PENALTY_CO2 = 5.0

    print("Start in-memory pseudo-LCA berekeningen...")

    # Stap A: Bereken Volume in m3 (dimensies zijn in mm, dus delen door 1000)
    df_lca['Volume_m3'] = (df_lca['Length_Actual'] / 1000) * (df_lca['Width'] / 1000) * (df_lca['Depth'] / 1000)

    # Stap B: Material Impact (A1-A3)
    df_lca['Impact_Material_kgCO2'] = df_lca['Volume_m3'] * df_lca['Embodied Carbon Coëfficiënt']

    # Stap C: Transport Impact (A4)
    # Dichtheid (Density) delen door 1000 om van kg/m3 naar ton/m3 te gaan
    df_lca['Impact_Transport_kgCO2'] = df_lca['Volume_m3'] * (df_lca['Density'] / 1000) * df_lca['Transport_Dist'] * df_lca['Emmisiefactor']

    # Stap D: Processing Impact (C3 / A3 Re-processing)
    # Binaire bewerkingsfactor (0 of 1) maal de vaste penalty
    df_lca['Impact_Processing_kgCO2'] = df_lca['Bewerkingsfactor'] * PROCESSING_PENALTY_CO2

    # Stap E: Totale E_cost Berekenen
    df_lca['E_cost_Total_kgCO2'] = (
        df_lca['Impact_Material_kgCO2'] +
        df_lca['Impact_Transport_kgCO2'] +
        df_lca['Impact_Processing_kgCO2']
    )

    # Afronden op 2 decimalen voor een overzichtelijke output
    cols_to_round = ['Volume_m3', 'Impact_Material_kgCO2', 'Impact_Transport_kgCO2', 'Impact_Processing_kgCO2', 'E_cost_Total_kgCO2']
    df_lca[cols_to_round] = df_lca[cols_to_round].round(2)

    print("Berekening voltooid! De dataset is verrijkt met LCA-scores.")
    return df_lca

# 1. Voer de functie uit op je reeds ingeladen DataFrame
df_stock_with_lca = calculate_pseudo_lca(df_input_csv)

# 2. Bekijk de resultaten van de berekening
print("\nPreview van de berekende E_cost per element:")
display(df_stock_with_lca.sample(10))

Start in-memory pseudo-LCA berekeningen...
Berekening voltooid! De dataset is verrijkt met LCA-scores.

Preview van de berekende E_cost per element:


,Member_ID,State,Length_Actual,Depth,Width,E_modulus_eff,f_mk,Density,Embodied Carbon Coëfficiënt,Transport_Dist,Emmisiefactor,Bewerkingsfactor,Volume_m3,Impact_Material_kgCO2,Impact_Transport_kgCO2,Impact_Processing_kgCO2,E_cost_Total_kgCO2
111,NS_00111,0,3300.0,300.0,100.0,11000.0,24,420,150.0,500,0.0842,0,0.10,14.85,1.75,0.0,16.60
179,RS_00012,1,5016.0,213.0,64.0,9000.0,18,380,15.0,64,0.0359,1,0.07,1.03,0.06,5.0,6.09
181,RS_00014,1,4070.0,186.0,190.0,11000.0,24,420,15.0,83,0.1142,1,0.14,2.16,0.57,5.0,7.73
35,NS_00035,0,2700.0,150.0,100.0,11000.0,24,420,150.0,500,0.1202,0,0.04,6.08,1.02,0.0,7.10
154,NS_00154,0,4000.0,200.0,75.0,11000.0,24,420,150.0,500,0.1028,0,0.06,9.00,1.30,0.0,10.30
112,NS_00112,0,3600.0,100.0,38.0,11000.0,24,420,150.0,500,0.1153,0,0.01,2.05,0.33,0.0,2.38
47,NS_00047,0,2700.0,225.0,100.0,11000.0,24,420,150.0,500,0.0841,0,0.06,9.11,1.07,0.0,10.19
156,NS_00156,0,4000.0,225.0,38.0,11000.0,24,420,150.0,500,0.0934,0,0.03,5.13,0.67,0.0,5.80
152,NS_00152,0,4000.0,200.0,38.0,11000.0,24,420,150.0,500,0.0984,0,0.03,4.56,0.63,0.0,5.19
7,NS_00007,0,2400.0,150.0,100.0,11000.0,24,420,150.0,500,0.1068,0,0.04,5.40,0.81,0.0,6.21


## Geometry

Omdat de pseudo-LCA berekening (uit de vorige stap) de impact van de gehele balk al berekent, heb je nu een methodologische keuze:

Optie A (De dubbele boete): Je telt de LCA-score én de penalty's bij elkaar op. Hiermee straf je afval extra zwaar af ten opzichte van het puur inbrengen van materiaal. Dit dwingt het algoritme agressief naar optimaal materiaalgebruik.

Optie B (De uitsplitsing): Je berekent de LCA-score alleen over het nuttige deel, en gebruikt je nieuwe formules om het verlies in kaart te brengen.

In [ ]:
# ==========================================
# CEL 1: INPUTS EN ALGEMENE PARAMETERS
# ==========================================
import pandas as pd
import numpy as np

# GWP Waarden (kg CO2 eq / kg hout)
GWP_VIRGIN = 0.50
GWP_RECLAIMED = 0.08

# LCA Parameters voor E_cost
PROCESSING_PENALTY_CO2 = 5.0  # kg CO2 boete voor bewerkingen (bijv. ontspijkeren)

print("✅ Parameters succesvol geladen.")

# MATCHING ALGORITHM / ILP

## Script

In [ ]:
import pulp

# 1. DATA DEFINITIE
# De rijen (Stock / X) en kolommen (Constructie / Y)
stock_items = ['X1', 'X2', 'X3', 'X4N', 'X5N']
construction_slots = ['Y1', 'Y2', 'Y3', 'Y4', 'Y5', 'Y6']

# Jouw Cost Matrix (zoals je die in de tabel gaf)
# Let op: de volgorde moet overeenkomen met stock_items
cost_matrix = [
    [2,  6,  8,  3,  1,  4],   # X1
    [4,  5,  3,  6,  3,  2],   # X2
    [1,  7,  4,  9,  6,  5],   # X3
    [10, 10, 2,  2,  10, 10],  # X4 (Nieuw)
    [10, 1,  1,  10, 10, 10]   # X5 (Nieuw)
]

# We maken een dictionary om makkelijk kosten op te zoeken: costs['X1']['Y1'] = 2
costs = {
    stock_items[i]: {construction_slots[j]: cost_matrix[i][j]
                     for j in range(len(construction_slots))}
    for i in range(len(stock_items))
}

# 2. HET MODEL OPZETTEN
# We willen de kosten MINIMALISEREN
prob = pulp.LpProblem("Reclaimed_Timber_Matching", pulp.LpMinimize)

# 3. DECISION VARIABLES
# Dit maakt voor elke combinatie X-Y een variabele die 0 of 1 kan zijn (Binary)
# x['X1']['Y1'] wordt 1 als we matchen, anders 0.
x = pulp.LpVariable.dicts("Match", (stock_items, construction_slots), 0, 1, pulp.LpBinary)

# 4. OBJECTIVE FUNCTION (Doel)
# Som van alle (kosten * keuze)
prob += pulp.lpSum([x[i][j] * costs[i][j] for i in stock_items for j in construction_slots])

# 5. CONSTRAINTS (De Regels)
# Regel A: Elke Y (constructie onderdeel) MOET precies 1 stuk hout krijgen.
for j in construction_slots:
    prob += pulp.lpSum([x[i][j] for i in stock_items]) == 1

# Regel B: Capaciteit van de Stock (X)
# X1, X2, X3 zijn 'reclaimed' en uniek -> Maximaal 1x gebruiken
reclaimed_limit = 1
for i in ['X1', 'X2', 'X3']:
    prob += pulp.lpSum([x[i][j] for j in construction_slots]) <= reclaimed_limit

# X4, X5 zijn 'nieuw' -> Mogen vaker gebruikt worden (bijv. max 6x, want er zijn 6 gaten)
new_limit = 10000
for i in ['X4N', 'X5N']:
    prob += pulp.lpSum([x[i][j] for j in construction_slots]) <= new_limit

# 6. OPLOSSEN
prob.solve()

# 7. RESULTAAT PRINTEN
print(f"Status: {pulp.LpStatus[prob.status]}")
print("-" * 35)

total_cost = 0
print(f"{'Stock':<10} -> {'Element':<10} {'Kosten'}")

for j in construction_slots:
    for i in stock_items:
        if x[i][j].varValue == 1: # Als de beslissing 'JA' is
            print(f"{j:<10} -> {i:<10} {costs[i][j]}")
            total_cost += costs[i][j]

print("-" * 35)
print(f"Totale 'Penalty' Cost: {total_cost}")